In [1]:
from tqdm.contrib.concurrent import process_map
from typing import *

/home/ddalton/Data/old_home/miniconda3/envs/scgpt_2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
"""Pre-Process Data

Convert the raw data counts into sc-RNAseq compatible data format.

Structure:
    1. Imports, Variables, Functions
    2. Load Data
    3. Convert to `adata` object
    4. Save to output file

"""

# region 1. Imports, Variables, Functions
# imports
import numpy as np, os, sys, pandas as pd, scanpy as sc
import anndata as ad
import logging
from tqdm import tqdm
from typing import *

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
from matplotlib import pyplot as plt
from datetime import datetime
import pickle
from typing import *
import json
import xml.etree.ElementTree as ET
import xml.etree.ElementTree as ET
import random
from tqdm.contrib.concurrent import process_map
from typing import *
from tqdm import tqdm
import os
import pandas as pd
import numpy as np
import networkx as nx
import logging
import json
import sys
import pickle
import obonet


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)


manual_parameters = {
    "dataset_exercise": "doid_dataset",
    "diseases_of_interest_set": None,
    "library_strategies_of_interest_set": list({"RNA-Seq", "Microarray"}),
    "ontology": "doid",
}


df_info_path = os.path.join(
    "/aloy",
    "home",
    "ddalton",
    "projects",
    "disease_signatures",
    "data",
    "DiSignAtlas",
    "Disease_information_Datasets_extended.csv",
)


large_df_path = "/aloy/home/ddalton/projects/disease_signatures/data/DiSignAtlas/DiSignAtlas.exp_prof_merged.csv"

base_output_dir = "../data"


mesh_file_path = os.path.join(
    "/aloy/home/ddalton/projects/disease_signatures/data/MeSH/desc2023.xml"
)


data_info_path = os.path.join(
    "/aloy/home/ddalton/projects/disease_signatures",
    "data",
    "DiSignAtlas",
    "Disease_information_Datasets.csv",
)

do_obl_data_path = os.path.join(
    "/aloy/home/ddalton/projects/disease_signatures",
    "data",
    "DiseaseOntology",
    "doid.obo",
)
external_links_data_path = os.path.join(
    "/aloy/home/ddalton/projects/disease_signatures",
    "data",
    "DiSignAtlas",
    "external_links.pkl",
)


# functions
def get_skip_rows(dsaids_interest):
    """Get Skip Rows
    Args:
        - dsaids_interest (list): List of DSAIDs of interest
    Returns:
        skip_rows_idxs (np.array): Array of indexes to skip
    """
    # variables
    large_df_path = "/aloy/home/ddalton/projects/disease_signatures/data/DiSignAtlas/DiSignAtlas.exp_prof_merged.csv"

    # load entire dataframe ID column only
    id_values = pd.read_csv(large_df_path, usecols=["ID"])["ID"].values

    # get indexes to skip
    skip_rows_idxs = np.argwhere(
        ~np.isin([x.split(";")[0] for x in id_values], dsaids_interest)
    ).flatten()

    skip_rows_idxs = skip_rows_idxs + 1  # add 1 to skip

    logging.info(f"Skipping {len(skip_rows_idxs)} rows")
    return skip_rows_idxs


def get_exp_prof(dsaids_interest):
    """Get Expression Profiles"""

    # variables
    file_dir = "/aloy/home/ddalton/projects/disease_signatures/data/DiSignAtlas/tmp/"
    first = True
    for dsaid in tqdm(dsaids_interest):
        __df = pd.read_csv(os.path.join(file_dir, f"{dsaid}.csv"))
        if first:
            df_global = __df
            first = False
        else:
            df_global = pd.concat([df_global, __df], axis=0)
    return df_global


def get_tissue(ids: List[str]) -> List[str]:
    """Get Tissue
    Args:
        - ids (list): List of IDs
    Returns:
        - tissues (list): List of tissues
    """
    dsaids = [x.split(";")[0] for x in ids]
    dsaid_2_tissue = dict(zip(df_info["dsaid"], df_info["tissue"]))
    tissues = [str(dsaid_2_tissue[dsaid]) for dsaid in dsaids]
    return tissues


def get_disease_study(ids: List[str]) -> List[str]:
    """Get Disease Study
    Args:
        - ids (list): List of IDs
    Returns:
        - diseases (list): List of diseases
    """
    dsaids = [x.split(";")[0] for x in ids]
    dsaid_2_disease = dict(zip(df_info["dsaid"], df_info["disease"]))
    disease_study = [str(dsaid_2_disease[dsaid]) for dsaid in dsaids]
    return disease_study


def get_disease(ids: List[str]) -> List[str]:
    """Get Disease
    Args:
        - ids (list): List of IDs
    Returns:
        - diseases (list): List of diseases
    """
    dsaid_2_disease = dict(zip(df_info["dsaid"], df_info["disease"]))
    diseases = list()
    for id in ids:
        dsaid = id.split(";")[0]
        state = id.split(";")[2]
        if state == "Control":
            diseases.append("Control")
        else:
            diseases.append(dsaid_2_disease.get(dsaid))
    return diseases


def get_mesh_disease(
    ids: List[str], mesh_id_2_term: dict, dsaid_2_mesh_id: dict
) -> List[str]:
    """Get MeSH Disease
    Args:
        - ids (list): List of IDs
    Returns:
        - mesh_diseases (list): List of diseases
    """
    mesh_diseases = list()
    mesh_ids = list()
    for id in ids:
        dsaid = id.split(";")[0]
        state = id.split(";")[2]
        if state == "Control":
            mesh_diseases.append("Control")
            mesh_ids.append("Control")
        else:
            mesh_id = dsaid_2_mesh_id.get(dsaid)[0]
            mesh_disease = mesh_id_2_term.get(mesh_id)

            mesh_diseases.append(mesh_disease)
            mesh_ids.append(mesh_id)

    return mesh_diseases, mesh_ids


def get_doid_disease(
    ids: List[str], doid_2_term: dict, dsaid_2_doid: dict
) -> List[str]:
    """Get DOID Disease
    Args:
        - ids (list): List of IDs
        - doid_2_term (dict): Dictionary with DOID to term
        - dsaid_2_doid (dict): Dictionary with DSAID to DOID
    Returns:
        - doid_diseases (list): List of diseases
    """
    doid_diseases = list()
    doid_ids = list()
    for id in ids:
        dsaid = id.split(";")[0]
        state = id.split(";")[2]
        if state == "Control":
            doid_diseases.append("Control")
            doid_ids.append("Control")
        else:
            doid_id = dsaid_2_doid.get(dsaid)[0]
            doid_disease = doid_2_term.get(doid_id)

            doid_diseases.append(doid_disease)
            doid_ids.append(doid_id)

    return doid_diseases, doid_ids


def get_dataset(ids: List[str]) -> List[str]:
    """Get Dataset
    Args:
        - ids (list): List of IDs
    Returns:
        - datasets (list): List of datasets
    """
    dsaids = [x.split(";")[0] for x in ids]
    dsaid_2_dataset = dict(zip(df_info["dsaid"], df_info["accession"]))
    datasets = [str(dsaid_2_dataset[dsaid]) for dsaid in dsaids]
    return datasets


def get_library(ids: List[str]) -> List[str]:
    """Get Library
    Args:
        - ids (list): List of IDs
    Returns:
        - datasets (list): List of datasets
    """
    dsaids = [x.split(";")[0] for x in ids]
    dsaid_2_dataset = dict(zip(df_info["dsaid"], df_info["library_strategy"]))
    datasets = [str(dsaid_2_dataset[dsaid]) for dsaid in dsaids]
    return datasets


def get_folder_name(base_output_dir: str) -> str:
    """Get Folder Name
    Args:
        - output_path (str): Output folder
    Returns:
        - output_dir (str): Output directory
    """
    # Step 1: Generate today's date string
    today = datetime.now().strftime("%y-%m-%d")

    # Step 2: Find the highest existing run number for today
    existing_runs = [
        d
        for d in os.listdir(base_output_dir)
        if os.path.isdir(os.path.join(base_output_dir, d))
        and d.startswith(f"pp_data-{today}")
    ]

    # Extract numbers from existing runs and find the max
    existing_numbers = [
        int(d.split("-")[-1]) for d in existing_runs if d.split("-")[-1].isdigit()
    ]

    # Calculate the next run number
    next_run_number = max(existing_numbers, default=0) + 1

    # Step 3: Create the directory name with zero-padded run number
    output_dir = os.path.join(base_output_dir, f"pp_data-{today}-{next_run_number:02d}")

    # Step 4: Create the directory
    os.makedirs(output_dir, exist_ok=True)

    print(f"Output directory created: {output_dir}")
    return output_dir


def get_dataset_to_batch(
    ids: List[str], df_info: pd.DataFrame
) -> Tuple[List[str], List[int]]:
    """Get Dataset to Batch
    Args:
        - ids (list): List of IDs
        - df_info (pd.DataFrame): DataFrame with information
    Returns:
        - dataset_accessions (list): List of dataset accessions
        - dataset_ids (list): List of dataset IDs
    """
    dsaid_2_accession = dict(zip(df_info["dsaid"], df_info["accession"]))

    dataset_accessions = [dsaid_2_accession[id.split(";")[0]] for id in ids]

    accession_2_id = {k: v for v, k in enumerate(set(dataset_accessions))}
    dataset_ids = [accession_2_id[accession] for accession in dataset_accessions]

    return dataset_accessions, dataset_ids


def get_diseases_n_datasets(df: pd.DataFrame, n: int = 10) -> List:
    """Get Diseases With More Than n Datasets

    Args:
        - df(pd.DataFrame): DataFrame with the information
        - diseases(List): List of diseases to filter
        - n(int): Number of datasets to filter

    Returns:
        - List: List of diseases with more than n datasets
    """
    diseases_list = list()
    dsaids_list = list()
    # iterate over diseases
    for disease in df["disease"].unique():
        df_query = df.query(f'disease == "{disease}"')
        if df_query["accession"].nunique() >= n:
            diseases_list.append(disease)
            dsaids_list.append(df_query["dsaid"].unique())

    return diseases_list, dsaids_list


def get_medium_dataset(df: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    """Get Medium Dataset

    Args:
        - df(pd.DataFrame): DataFrame with the information
        - n(int): Number of datasets to filter

    Returns:
        - pd.DataFrame: DataFrame with the medium dataset
    """
    # load mappings dsaids -> MeSH terms
    mesh_terms = pickle.load(
        open(
            "/aloy/home/ddalton/projects/disease_signatures/data/DiSignAtlas/mesh_tree_terms.pkl",
            "rb",
        )
    )

    dsaid_2_mesh = {
        k: v for k, v in zip(mesh_terms["dsaids"], mesh_terms["mesh_tree_terms"])
    }

    dsaids_with_mesh = [k for k, v in dsaid_2_mesh.items() if len(v) > 0]

    # filter by nº of datasets
    diseases_list, dsaids_list = get_diseases_n_datasets(df, n)

    # filter by MeSH term presence
    diseases_f_mesh = list()
    dsaids_f_mesh = list()
    for disease_i, dsaids_i in tqdm(
        zip(diseases_list, dsaids_list), total=len(diseases_list)
    ):

        if np.isin(dsaids_i, dsaids_with_mesh).any():
            diseases_f_mesh.append(disease_i)
            for dsaid_j in dsaids_i:
                if dsaid_j in dsaids_with_mesh:
                    dsaids_f_mesh.append(dsaid_j)
                else:
                    logging.info(
                        "DSAID of a disease w/ other DSAID w/ MeSH terms - but it itself doesn't have MeSH terms"
                    )
                    logging.info(f"{dsaid_j}, {disease_i}")

    logging.info(f"Nº of diseases {len(diseases_f_mesh)}/{len(diseases_list)}")
    logging.info(
        f"Nº of dsaids {len(dsaids_f_mesh)}/{len([x for sublist in dsaids_list for x in sublist])}"
    )
    return diseases_f_mesh, dsaids_f_mesh


def parse_mesh_data_with_ids(file_path):
    """Parse MeSH XML data and extract disease terms along with MeSH IDs."""
    tree = ET.parse(file_path)
    root = tree.getroot()

    # Dictionaries to map tree numbers and MeSH IDs to disease terms
    tree_2_term = {}
    term_2_tree = {}
    mesh_id_2_term = {}

    # Extract disease terms, tree numbers, and MeSH IDs
    for descriptor in root.findall("DescriptorRecord"):
        # Get the disease term name
        term = descriptor.find("DescriptorName/String").text

        # Get the MeSH ID
        mesh_id = descriptor.find("DescriptorUI").text
        mesh_id_2_term[mesh_id] = term

        # Get all tree numbers for this term
        tree_numbers = descriptor.findall("TreeNumberList/TreeNumber")

        for tree_number in tree_numbers:
            # Map each tree number to its term
            tree_2_term[tree_number.text] = term
            term_2_tree[term] = tree_number.text

    return tree_2_term, term_2_tree, mesh_id_2_term


def parse_mesh_data_with_ids(file_path):
    """Parse MeSH XML data and extract disease terms along with MeSH IDs."""
    tree = ET.parse(file_path)
    root = tree.getroot()

    # Dictionaries to map tree numbers and MeSH IDs to disease terms
    tree_2_term = {}
    term_2_tree = {}
    mesh_id_2_term = {}

    # Extract disease terms, tree numbers, and MeSH IDs
    for descriptor in root.findall("DescriptorRecord"):
        # Get the disease term name
        term = descriptor.find("DescriptorName/String").text

        # Get the MeSH ID
        mesh_id = descriptor.find("DescriptorUI").text
        mesh_id_2_term[mesh_id] = term

        # Get all tree numbers for this term
        tree_numbers = descriptor.findall("TreeNumberList/TreeNumber")

        for tree_number in tree_numbers:
            # Map each tree number to its term
            tree_2_term[tree_number.text] = term
            term_2_tree[term] = tree_number.text

    return tree_2_term, term_2_tree, mesh_id_2_term


def shorten_terms(mesh_terms: List) -> List:
    """Shorten Terms
    Args:
        - mesh_terms(List): List of MeSH terms
    Returns:
        - List: List of shortened MeSH terms
    """
    return [x.split(".")[0] for x in mesh_terms]


def get_unrelated_dsaids_from_dataset(dsaids: List, dsaid_2_tree_terms: dict) -> Set:
    """Check if 2+ dsaids from the same dataset are unrelated!
    Args:
        - mesh_id_1(str): MeSH ID 1
        - mesh_id_2(str): MeSH ID 2
        - dsaid_2_tree_terms(dict): Dictionary with MeSH ID to term
    Returns:
        - Set: MeSH IDs that are unrelated
    """
    unrelated_dsaids = set()
    n_dsaids = len(dsaids)
    for i in range(n_dsaids):
        for j in range(i + 1, n_dsaids):
            dsaid_i = dsaids[i]
            dsaid_j = dsaids[j]

            # get tree terms
            term_i = dsaid_2_tree_terms.get(dsaid_i)
            term_j = dsaid_2_tree_terms.get(dsaid_j)

            # exclude overly general terms - i.e don't want "Neoplasm"
            term_i = [x for x in term_i if len(x.split(".")) > 1]
            term_j = [x for x in term_j if len(x.split(".")) > 1]

            # shorten terms to simplify comparisons
            term_i = shorten_terms(term_i)
            term_j = shorten_terms(term_j)

            # check no related terms between
            if len(set(term_i).intersection(set(term_j))) == 0:
                unrelated_dsaids.add(dsaid_i)
                unrelated_dsaids.add(dsaid_j)

    return unrelated_dsaids


def get_unrelated_dsaids_from_all(df: pd.DataFrame, dsaid_2_tree_terms: dict) -> List:
    """Get unrelated dsaids from all datasets
    Args:
        - df(pd.DataFrame): DataFrame with the information
    Returns:
        - List: List of unrelated dsaids
    """
    # Get datasets which have 1+ diseases
    df = df.groupby("accession").filter(lambda x: x["mesh_id"].nunique() >= 2)

    unrelated_dsaids = set()
    for dataset_i in df["accession"].unique():
        QUERY = f"accession == '{dataset_i}'"
        dsaids_i = df.query(QUERY)["dsaid"].to_list()
        unrelated_dsaids_i = get_unrelated_dsaids_from_dataset(
            dsaids_i, dsaid_2_tree_terms
        )
        unrelated_dsaids.update(unrelated_dsaids_i)

    return list(unrelated_dsaids)


def get_bias_dataset(
    df_filtered: pd.DataFrame,
    dsaid_2_tree_terms: dict,
    dsaid_2_mesh_id: dict,
    mesh_id_2_term: dict,
) -> pd.DataFrame:
    """Generate a dataset which allows us to asses dataset biases.
    To do so we select datasets that have 2+ unrelated diseases.
    For completeness we  include for those diseases w/ less than 5 datasets additional datasets.

    Args:
        - df_info(pd.DataFrame): DataFrame with the information
        - dsaid_2_tree_terms(dict): Dictionary with MeSH ID to term
        - dsaid_2_mesh_id(dict): Dictionary with MeSH ID to term
        - mesh_id_2_term(dict): Dictionary with MeSH ID to term
    Returns:
        - pd.DataFrame: DataFrame with the information
    """

    logging.info(df_info.shape)

    # filter by nº of samples
    n_samples_upp_thr = 200
    n_samples_low_thr = 10

    # Filter by nº of samples
    # add nº of samples - must have both control & case
    df_filtered = df_filtered.copy()
    df_filtered["n_samples"] = [
        (
            len(x.split(";")) + len(y.split(";"))
            if isinstance(x, str) and isinstance(y, str)
            else 0
        )
        for x, y in zip(df_filtered["Control"], df_filtered["Case"])
    ]

    # filter
    QUERY = f"n_samples >= {n_samples_low_thr} & n_samples <= {n_samples_upp_thr}"
    df_query = df_filtered.query(QUERY)
    logging.info(f"Filter: nº of samples: {df_query.shape[0]}")

    # Filter by presence of Disease MeSH IDs
    # mask
    mask_mesh_tree = list()
    for dsaid_i in df_query["dsaid"]:
        mesh_tree_i = dsaid_2_tree_terms.get(dsaid_i)
        if mesh_tree_i is not None:
            # presence of disease tree term
            mesh_tree_i = [x for x in mesh_tree_i if x.startswith("C")]
            if len(mesh_tree_i) > 0:
                mask_mesh_tree.append(True)
            else:
                mask_mesh_tree.append(False)
        else:
            mask_mesh_tree.append(False)

    df_query = df_query[mask_mesh_tree]
    logging.info(f"Filter: MeSH Tree Disease presence {df_query.shape[0]}")

    # add mesh id
    df_query["mesh_id"] = [
        dsaid_2_mesh_id.get(x)[0] if len(dsaid_2_mesh_id.get(x)) > 0 else np.nan
        for x in df_query["dsaid"]
    ]
    df_query = df_query.dropna(subset=["mesh_id"])
    logging.info(f"Filter by mesh_ids {df_query.shape[0]}")

    # add mesh id info
    df_query["mesh_disease"] = [mesh_id_2_term.get(x) for x in df_query["mesh_id"]]

    # Filter diseases present in 5+ datasets
    # Group by disease
    disease_counts = df_query.groupby("mesh_id")["accession"].nunique()

    # Filter for diseases that are in 5 or more unique datasets
    disease_counts = disease_counts[disease_counts >= 5]
    df_query = df_query[df_query["mesh_id"].isin(disease_counts.index)]
    logging.info(f"Filter 5+ datasets for each disease: {df_query.shape[0]}")

    # Get dsaids with another dsaid from an unrelated disease in the same dataset
    unrelated_dsaids = get_unrelated_dsaids_from_all(df_query, dsaid_2_tree_terms)
    logging.info(f"Nº of unrelated dsaids {len(unrelated_dsaids)}")

    # filter df_query_2 by unrelated dsaids
    df_unrelated = df_query[df_query["dsaid"].isin(unrelated_dsaids)]
    logging.info(f"Filter: Unrelated diseases {df_unrelated.shape[0]}")

    # filter out all datasets that have unrelated diseases
    df_rest = df_query[~df_query["accession"].isin(df_unrelated["accession"])]

    # only have 1 dsaid for each disease
    df_rest = df_rest.copy()
    df_rest["mesh_id_accession"] = [
        x + "_" + y for x, y in zip(df_rest["mesh_id"], df_rest["accession"])
    ]
    df_rest.drop_duplicates(subset=["mesh_id_accession"], inplace=True)

    # check which diseases have less than 5 datasets/accessions
    dsaids_rest = set()
    for mesh_id_i in df_unrelated["mesh_id"].unique():
        accessions_i = df_unrelated.query(f"mesh_id == '{mesh_id_i}'")[
            "accession"
        ].unique()
        n_datasets_i = len(accessions_i)
        if n_datasets_i < 5:
            n_samples_i = 5 - n_datasets_i
            dsaids_i = df_rest.query(f"mesh_id == '{mesh_id_i}'")["dsaid"].to_list()
            remaining_samples_i = set(dsaids_i) - dsaids_rest
            sample_i = random.sample(list(remaining_samples_i), n_samples_i)
            dsaids_rest.update(sample_i)
        else:
            logging.info(f"MeSH id {mesh_id_i} has {n_datasets_i} datasets")

    logging.info(f"DSAIDs to sample {len(dsaids_rest)}")

    # get dsaids for diseases with less than 5
    df_final = pd.concat([df_unrelated, df_rest.query("dsaid in @dsaids_rest")])
    logging.info(f"Final shape {df_final.shape[0]}")
    return df_final


def get_data_leakage_dataset(
    df_filtered: pd.DataFrame,
    dsaid_2_tree_terms: dict,
    dsaid_2_mesh_id: dict,
    mesh_id_2_term: dict,
    thr_n_datasets=20,
) -> pd.DataFrame:
    """Generate a dataset which allows us to asses how much our model can generalize to new datasets.
    To do so we select diseases which appear in 20+ datasets..

    Args:
        - df_info(pd.DataFrame): DataFrame with the information
        - dsaid_2_tree_terms(dict): Dictionary with MeSH ID to term
        - dsaid_2_mesh_id(dict): Dictionary with MeSH ID to term
        - mesh_id_2_term(dict): Dictionary with MeSH ID to term
    Returns:
        - pd.DataFrame: DataFrame with the information
    """

    logging.info(df_info.shape)

    # filter by nº of samples
    n_samples_upp_thr = 100
    n_samples_low_thr = 10

    # Filter by nº of samples
    # add nº of samples - must have both control & case
    df_filtered = df_filtered.copy()
    df_filtered["n_samples"] = [
        (
            len(x.split(";")) + len(y.split(";"))
            if isinstance(x, str) and isinstance(y, str)
            else 0
        )
        for x, y in zip(df_filtered["Control"], df_filtered["Case"])
    ]

    # filter
    QUERY = f"n_samples >= {n_samples_low_thr} & n_samples <= {n_samples_upp_thr}"
    df_query = df_filtered.query(QUERY)
    logging.info(f"Filter: nº of samples: {df_query.shape[0]}")

    # Filter by presence of Disease MeSH IDs
    # mask
    mask_mesh_tree = list()
    for dsaid_i in df_query["dsaid"]:
        mesh_tree_i = dsaid_2_tree_terms.get(dsaid_i)
        if mesh_tree_i is not None:
            # presence of disease tree term
            mesh_tree_i = [x for x in mesh_tree_i if x.startswith("C")]
            if len(mesh_tree_i) > 0:
                mask_mesh_tree.append(True)
            else:
                mask_mesh_tree.append(False)
        else:
            mask_mesh_tree.append(False)

    df_query = df_query[mask_mesh_tree]
    logging.info(f"Filter: MeSH Tree Disease presence {df_query.shape[0]}")

    # add mesh id
    df_query["mesh_id"] = [
        dsaid_2_mesh_id.get(x)[0] if len(dsaid_2_mesh_id.get(x)) > 0 else np.nan
        for x in df_query["dsaid"]
    ]
    df_query = df_query.dropna(subset=["mesh_id"])
    logging.info(f"Filter by mesh_ids {df_query.shape[0]}")

    # add mesh id info
    df_query["mesh_disease"] = [mesh_id_2_term.get(x) for x in df_query["mesh_id"]]

    # Filter diseases present in n+ datasets
    # Group by disease
    disease_counts = df_query.groupby("mesh_id")["accession"].nunique()

    # Filter for diseases that are in n+ unique datasets
    disease_counts = disease_counts[disease_counts >= thr_n_datasets]
    df_final = df_query[df_query["mesh_id"].isin(disease_counts.index)]
    logging.info(
        f"Filter {thr_n_datasets}+ datasets for each disease: {df_query.shape[0]}"
    )

    logging.info(f"Nº unique diseases: {df_final['mesh_id'].nunique()}")
    logging.info(f"Nº unique datasets: {df_final['accession'].nunique()}")

    return df_final


# functions


def load_do_graph():
    """Load the Disease Ontology graph from the OBO file."""
    do_obl_data_path = os.path.join(
        "/aloy/home/ddalton/projects/disease_signatures",
        "data",
        "DiseaseOntology",
        "doid.obo",
    )
    do_graph = obonet.read_obo(do_obl_data_path)
    return nx.DiGraph(do_graph)


def get_processed_ids():
    """Get processed ids
    Returns:
        list: list of processed ids
    """
    data_path = os.path.join(
        "/aloy/home/ddalton/projects/disease_signatures",
        "data",
        "DiSignAtlas",
        "dsa_diff_download.processed",
    )
    return [f.split("_")[0] for f in os.listdir(data_path)]


# get all entrez protein-coding human ids
def get_human_entrez_protein_coding_ids():
    """Get Human Entrez IDs
    Returns:
        list: list of human entrez ids
    """
    data_path = os.path.join(
        "/aloy/home/ddalton/projects/disease_signatures",
        "data",
        "ncbi_gene_info",
        "gene_info",
    )
    df = pd.read_csv(data_path, sep="\t", usecols=["#tax_id", "GeneID", "type_of_gene"])
    df_human = df[(df["#tax_id"] == 9606) & (df["type_of_gene"] == "protein-coding")]
    logging.info(f"Nº of Human protein coding genes: {len(df_human)}")
    return df_human["GeneID"].to_list()


def get_de_genes(
    signature: List, thr_log2FC: float = 0, thr_p_val: float = 0.05
) -> tuple[List, List]:
    """
    Get DE genes for each signature
    Args:
        - thr_log2FC: float
            Threshold for log2 fold change
        - thr_p_val: float
            Threshold for p-value

    Returns:
        - upregulated genes: list
        - downregulated genes: list
    """
    genes = np.array(signature[2])
    log2fc = np.array(signature[5])
    p_values = np.array(signature[4])

    global human_entrez_protein_coding_ids

    # filter by human genes
    mask_human = np.isin(np.array(genes), human_entrez_protein_coding_ids)
    genes = np.array(genes)[mask_human]
    p_values = np.array(p_values)[mask_human]
    log2fc = np.array(signature[5])[mask_human]
    total_genes = len(genes)

    # filter by thresholds
    mask_upr = (log2fc >= thr_log2FC) & (p_values <= thr_p_val)
    mask_dwn = (log2fc <= -thr_log2FC) & (p_values <= thr_p_val)

    return genes[mask_upr], genes[mask_dwn], total_genes


def get_dataset_do(signatures, do_G, data_info_path, dsaids_2_doids):
    """
    This fuction compiles filtering steps to get final list of dsaids
    Steps:
        1. Filter by human genes
        2. Filter by library strategy
        3. Filter by presence of disease ontology ids

    """

    # Load data

    processed_ids = get_processed_ids()
    logging.info(f"Nº of processed ids: {len(processed_ids)}")

    df_data_info = pd.read_csv(data_info_path)

    df_data_info_processed = df_data_info.copy()

    df_data_info_processed = df_data_info_processed[
        df_data_info_processed["dsaid"].isin(processed_ids)
    ]

    logging.info(f"Nº of processed ids in df_data_info: {len(df_data_info_processed)}")

    df_data_info_processed_filtered = df_data_info_processed[
        (df_data_info_processed["organism"] == "Homo sapiens")
        & (
            (df_data_info_processed["library_strategy"] == "Microarray")
            | (df_data_info_processed["library_strategy"] == "RNA-Seq")
        )
    ]
    logging.info(
        f"Nº of Filtered by library (filter out single cell): {df_data_info_processed_filtered.shape}"
    )

    # get all entrez protein-coding human ids
    global human_entrez_protein_coding_ids

    # Filter 1: Human Genes
    human_dsaids = df_data_info_processed_filtered["dsaid"].to_list()

    pct_thr = 0.5
    # even though we have filtered by human, we now look at the actual genes! Some may not be human !
    pct_human = list()
    dsaids_f1 = list()
    for i in range(len(signatures)):
        if signatures[i][0] in human_dsaids:
            _pct = len(
                set(signatures[i][2]).intersection(human_entrez_protein_coding_ids)
            ) / len(set(signatures[i][2]))
            pct_human.append(_pct)
            if _pct > pct_thr:
                dsaids_f1.append(signatures[i][0])
    pct_human = np.array(pct_human)

    df_data_info_processed_filtered = df_data_info_processed_filtered.copy()
    df_data_info_processed_filtered["n_samples"] = [
        int(x.split("|")[1])
        for x in df_data_info_processed_filtered["control_case_sample_count"]
    ]

    df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_f1")

    print(f"Nº of signatures: {len(df_query)}")
    print(f"Nº of datasets : {len(df_query['accession'].unique())}")
    print(f"Nº of diseases : {len(df_query['disease'].unique())}")
    print(f"Nº of samples : {df_query['n_samples'].sum()}")

    # Filter 2: Type of Sequencing
    df_data_info_processed_filtered["library_strategy"].value_counts()
    dsaids_f2 = dsaids_f1

    # Filter 3: Presence of Disease Ontology IDs
    # Leaf nodes Diseases
    leaf_nodes = [
        node
        for node in do_G.nodes()
        if do_G.in_degree(node) == 0 and do_G.out_degree(node) > 0
    ]

    dsaids_w_doids = [d for d in dsaids_f2 if d in dsaids_2_doids.keys()]
    dsaids_w_diseases = [
        d for d in dsaids_w_doids if len(set(dsaids_2_doids[d]) & set(leaf_nodes)) > 0
    ]

    # for now only take DSAIDS with 1 LEAF DISEASE!
    dsaids_f3 = [
        d
        for d in dsaids_w_diseases
        if len(set(dsaids_2_doids[d]) & set(leaf_nodes)) == 1
    ]
    print(f"Nº of DSAIDS with 1 leaf disease {len(dsaids_f3)}")

    # Filter 3.1: Presence of Disease Ontology IDs
    flatten = lambda l: [item for sublist in l for item in sublist]
    df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_w_doids")
    print(f"Filtering 3.1 - has doid")
    print(f"Nº of signatures: {len(df_query)}")
    print(f"Nº of datasets : {len(df_query['accession'].unique())}")
    print(f"Nº of diseases : {len(df_query['disease'].unique())}")
    print(
        f"Nº of disease ontology ids: {len(set(flatten([dsaids_2_doids[x] for x in dsaids_w_doids])))}"
    )
    print(
        f"Nº of disease ontology diseases: {len(set(flatten([list(set(dsaids_2_doids[x])&set(leaf_nodes)) for x in dsaids_w_doids])))}"
    )
    print(f"Nº of samples : {df_query['n_samples'].sum()}")

    # Filter 3.2: Presence of Leaf Disease Ontology IDs
    df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_w_diseases")
    print(f"Filtering 3.2 - has doid leaf")
    print(f"Nº of signatures: {len(df_query)}")
    print(f"Nº of datasets : {len(df_query['accession'].unique())}")
    print(f"Nº of diseases : {len(df_query['disease'].unique())}")
    print(
        f"Nº of disease ontology ids: {len(set(flatten([dsaids_2_doids[x] for x in dsaids_w_diseases])))}"
    )
    print(
        f"Nº of disease ontology diseases: {len(set(flatten([list(set(dsaids_2_doids[x])&set(leaf_nodes)) for x in dsaids_w_diseases])))}"
    )
    print(f"Nº of samples : {df_query['n_samples'].sum()}")

    # Filter 3.3: Presence of 1 Leaf Disease Ontology IDs
    df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_f3")
    print(f"Filtering 3.3 - has 1 doid leaf")
    print(f"Nº of signatures: {len(df_query)}")
    print(f"Nº of datasets : {len(df_query['accession'].unique())}")
    print(f"Nº of diseases : {len(df_query['disease'].unique())}")
    print(
        f"Nº of disease ontology ids: {len(set(flatten([dsaids_2_doids[x] for x in dsaids_f3])))}"
    )
    print(
        f"Nº of disease ontology diseases: {len(set(flatten([list(set(dsaids_2_doids[x])&set(leaf_nodes)) for x in dsaids_f3])))}"
    )
    print(f"Nº of samples : {df_query['n_samples'].sum()}")

    # Filter 4: Presence of DE Genes
    f_signatures = [s for s in signatures if s[0] in dsaids_f3]
    de_genes = process_map(get_de_genes, f_signatures, max_workers=8, chunksize=10)

    # Filter 4.1: Presence of DE Genes
    # Less than 50% of genes DE
    # at least 50 DE genes
    mask_de_genes = np.array(
        [(True if 0 <= len(d[0]) + len(d[1]) <= 1 * d[2] else False) for d in de_genes]
    )

    # get dsaids which pass filter
    dsaids_f4 = np.array(dsaids_f3)[mask_de_genes]
    print(f"Filtered by nº of DE genes: {len(dsaids_f4)}")

    df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_f4")
    print(f"Filtering 4 - has 1 doid leaf")
    print(f"Nº of signatures: {len(df_query)}")
    print(f"Nº of datasets : {len(df_query['accession'].unique())}")
    print(f"Nº of diseases : {len(df_query['disease'].unique())}")
    print(
        f"Nº of disease ontology ids: {len(set(flatten([dsaids_2_doids[x] for x in dsaids_f4])))}"
    )
    print(
        f"Nº of disease ontology diseases: {len(set(flatten([list(set(dsaids_2_doids[x])&set(leaf_nodes)) for x in dsaids_f4])))}"
    )
    print(f"Nº of samples : {df_query['n_samples'].sum()}")

    return dsaids_f4



In [3]:

# endregion

# region 2. Load Data
diseases_of_interest_set = manual_parameters.get("diseases_of_interest_set")
library_strategies_of_interest_set = manual_parameters.get(
    "library_strategies_of_interest_set"
)

# load DataFtame Info
df_info = pd.read_csv(df_info_path)


# Load MeSH data mappings
tree_2_term, term_2_tree, mesh_id_2_term = parse_mesh_data_with_ids(
    file_path=mesh_file_path
)

mesh_terms = pickle.load(
    open(
        "/aloy/home/ddalton/projects/disease_signatures/data/DiSignAtlas/mesh_tree_terms.pkl",
        "rb",
    )
)
dsaid_2_mesh_id = dict(zip(mesh_terms["dsaids"], mesh_terms["mesh_ids"]))
dsaid_2_tree_terms = dict(zip(mesh_terms["dsaids"], mesh_terms["mesh_tree_terms"]))


# Load DO data mappings
url = "http://purl.obolibrary.org/obo/doid.obo"
do_graph = obonet.read_obo(url)
do_G = nx.DiGraph(do_graph)

# Generate a mapping of DOID to its term (disease name)
doid_2_term = {
    node: data["name"] for node, data in do_G.nodes(data=True) if "name" in data
}

# get leaf nodes
leaf_nodes = [
    node
    for node in do_G.nodes()
    if do_G.in_degree(node) == 0 and do_G.out_degree(node) > 0
]

external_links_data_path = os.path.join(
    "/aloy/home/ddalton/projects/disease_signatures",
    "data",
    "DiSignAtlas",
    "external_links.pkl",
)

# load external links
with open(external_links_data_path, "rb") as f:
    external_links = pickle.load(f)
print(f"Loaded {len(external_links)} DiSignAtlas metadata (external links)")

# get DOID for each dsa
dsaids_2_doids = {}
for dsaid, external_link in zip(
    external_links["dsaids"], external_links["external_links"]
):
    do_ids = [
        link.replace("DO:", "DOID:") for link in external_link if link.startswith("DO")
    ]
    if len(do_ids) > 0:
        dsaids_2_doids[dsaid] = do_ids

# get DOID Leaf for each dsaid
dsaids_2_doids_leaf = {
    k: list(set(v) & set(leaf_nodes)) for k, v in dsaids_2_doids.items()
}


# filter by library strategy
QUERY = f"library_strategy in @library_strategies_of_interest_set & organism == 'Homo sapiens'"
df_filtered = df_info.query(QUERY)


Loaded 2 DiSignAtlas metadata (external links)


In [4]:
        # get human entrez ids
        human_entrez_protein_coding_ids = get_human_entrez_protein_coding_ids()

        # load signatures
        path_pkl = os.path.join(
            "/aloy/home/ddalton/projects/disease_signatures/",
            "data",
            "DiSignAtlas",
            "signatures.pkl",
        )
        logging.info(f"Loading signatures from file {path_pkl}")
        signatures = pickle.load(open(path_pkl, "rb"))

2025-08-02 16:45:28,531 - Nº of Human protein coding genes: 20608
2025-08-02 16:45:28,586 - Loading signatures from file /aloy/home/ddalton/projects/disease_signatures/data/DiSignAtlas/signatures.pkl


In [ ]:


if manual_parameters.get("dataset_exercise"):
    if manual_parameters["dataset_exercise"] == "small":
        print("Small Dataset")

    elif manual_parameters["dataset_exercise"] == "medium":
        print("Medium Dataset")

        # get dsaids of interest
        diseases_interest, dsaids_interest = get_medium_dataset(df_filtered, 10)

        df = get_exp_prof(dsaids_interest)

    elif manual_parameters["dataset_exercise"] == "data_leakage":
        print("Data Leakage Dataset")

        df_dl_dataset = get_data_leakage_dataset(
            df_filtered, dsaid_2_tree_terms, dsaid_2_mesh_id, mesh_id_2_term
        )

        dsaids_interest = df_dl_dataset["dsaid"].to_list()

        df = get_exp_prof(dsaids_interest)

    elif manual_parameters["dataset_exercise"] == "bias_dataset":

        # get dsaids of interest
        df_bias = get_bias_dataset(
            df_filtered, dsaid_2_tree_terms, dsaid_2_mesh_id, mesh_id_2_term
        )
        dsaids_interest = list(df_bias["dsaid"].unique())

        df = get_exp_prof(dsaids_interest)

    elif manual_parameters["dataset_exercise"] == "large":
        print("Large Dataset")
    elif manual_parameters["dataset_exercise"] == "doid_dataset":
        print("DOID Dataset")
        # get human entrez ids
        human_entrez_protein_coding_ids = get_human_entrez_protein_coding_ids()

        # load signatures
        path_pkl = os.path.join(
            "/aloy/home/ddalton/projects/disease_signatures/",
            "data",
            "DiSignAtlas",
            "signatures.pkl",
        )
        logging.info(f"Loading signatures from file {path_pkl}")
        signatures = pickle.load(open(path_pkl, "rb"))

        dsaids_interest = get_dataset_do(
            signatures, do_G, data_info_path, dsaids_2_doids
        )
        df = get_exp_prof(dsaids_interest)


# if specific diseases
else:
    QUERY = "disease in @diseases_of_interest_set & library_strategy in @library_strategies_of_interest_set & organism == 'Homo sapiens'"
    dsaids_interest = df_info.query(QUERY)["dsaid"].to_list()
    # df = get_exp_prof(dsaids_interest)
    df = get_exp_prof(dsaids_interest)


logging.info(f"Nº of DSAIDs of interest: {len(dsaids_interest)}")


# load dataframe
logging.info(f"Loaded dataframe with shape: {df.shape}")

In [5]:
# isolating function

# Load data

processed_ids = get_processed_ids()
logging.info(f"Nº of processed ids: {len(processed_ids)}")

df_data_info = pd.read_csv(data_info_path)

df_data_info_processed = df_data_info.copy()

df_data_info_processed = df_data_info_processed[
    df_data_info_processed["dsaid"].isin(processed_ids)
]

logging.info(f"Nº of processed ids in df_data_info: {len(df_data_info_processed)}")

df_data_info_processed_filtered = df_data_info_processed[
    (df_data_info_processed["organism"] == "Homo sapiens")
    & (
        (df_data_info_processed["library_strategy"] == "Microarray")
        | (df_data_info_processed["library_strategy"] == "RNA-Seq")
    )
]
logging.info(
    f"Nº of Filtered by library (filter out single cell): {df_data_info_processed_filtered.shape}"
)

# get all entrez protein-coding human ids
global human_entrez_protein_coding_ids

# Filter 1: Human Genes
human_dsaids = df_data_info_processed_filtered["dsaid"].to_list()

pct_thr = 0.5
# even though we have filtered by human, we now look at the actual genes! Some may not be human !
pct_human = list()
dsaids_f1 = list()
for i in range(len(signatures)):
    if signatures[i][0] in human_dsaids:
        _pct = len(
            set(signatures[i][2]).intersection(human_entrez_protein_coding_ids)
        ) / len(set(signatures[i][2]))
        pct_human.append(_pct)
        if _pct > pct_thr:
            dsaids_f1.append(signatures[i][0])
pct_human = np.array(pct_human)

df_data_info_processed_filtered = df_data_info_processed_filtered.copy()
df_data_info_processed_filtered["n_samples"] = [
    int(x.split("|")[1])
    for x in df_data_info_processed_filtered["control_case_sample_count"]
]

df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_f1")

print(f"Nº of signatures: {len(df_query)}")
print(f"Nº of datasets : {len(df_query['accession'].unique())}")
print(f"Nº of diseases : {len(df_query['disease'].unique())}")
print(f"Nº of samples : {df_query['n_samples'].sum()}")

# Filter 2: Type of Sequencing
df_data_info_processed_filtered["library_strategy"].value_counts()
dsaids_f2 = dsaids_f1

# Filter 3: Presence of Disease Ontology IDs
# Leaf nodes Diseases
leaf_nodes = [
    node
    for node in do_G.nodes()
    if do_G.in_degree(node) == 0 and do_G.out_degree(node) > 0
]

dsaids_w_doids = [d for d in dsaids_f2 if d in dsaids_2_doids.keys()]
dsaids_w_diseases = [
    d for d in dsaids_w_doids if len(set(dsaids_2_doids[d]) & set(leaf_nodes)) > 0
]

# for now only take DSAIDS with 1 LEAF DISEASE!
dsaids_f3 = [
    d
    for d in dsaids_w_diseases
    if len(set(dsaids_2_doids[d]) & set(leaf_nodes)) == 1
]
print(f"Nº of DSAIDS with 1 leaf disease {len(dsaids_f3)}")

# Filter 3.1: Presence of Disease Ontology IDs
flatten = lambda l: [item for sublist in l for item in sublist]
df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_w_doids")
print(f"Filtering 3.1 - has doid")
print(f"Nº of signatures: {len(df_query)}")
print(f"Nº of datasets : {len(df_query['accession'].unique())}")
print(f"Nº of diseases : {len(df_query['disease'].unique())}")
print(
    f"Nº of disease ontology ids: {len(set(flatten([dsaids_2_doids[x] for x in dsaids_w_doids])))}"
)
print(
    f"Nº of disease ontology diseases: {len(set(flatten([list(set(dsaids_2_doids[x])&set(leaf_nodes)) for x in dsaids_w_doids])))}"
)
print(f"Nº of samples : {df_query['n_samples'].sum()}")

# Filter 3.2: Presence of Leaf Disease Ontology IDs
df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_w_diseases")
print(f"Filtering 3.2 - has doid leaf")
print(f"Nº of signatures: {len(df_query)}")
print(f"Nº of datasets : {len(df_query['accession'].unique())}")
print(f"Nº of diseases : {len(df_query['disease'].unique())}")
print(
    f"Nº of disease ontology ids: {len(set(flatten([dsaids_2_doids[x] for x in dsaids_w_diseases])))}"
)
print(
    f"Nº of disease ontology diseases: {len(set(flatten([list(set(dsaids_2_doids[x])&set(leaf_nodes)) for x in dsaids_w_diseases])))}"
)
print(f"Nº of samples : {df_query['n_samples'].sum()}")

# Filter 3.3: Presence of 1 Leaf Disease Ontology IDs
df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_f3")
print(f"Filtering 3.3 - has 1 doid leaf")
print(f"Nº of signatures: {len(df_query)}")
print(f"Nº of datasets : {len(df_query['accession'].unique())}")
print(f"Nº of diseases : {len(df_query['disease'].unique())}")
print(
    f"Nº of disease ontology ids: {len(set(flatten([dsaids_2_doids[x] for x in dsaids_f3])))}"
)
print(
    f"Nº of disease ontology diseases: {len(set(flatten([list(set(dsaids_2_doids[x])&set(leaf_nodes)) for x in dsaids_f3])))}"
)
print(f"Nº of samples : {df_query['n_samples'].sum()}")

# Filter 4: Presence of DE Genes
f_signatures = [s for s in signatures if s[0] in dsaids_f3]
de_genes = process_map(get_de_genes, f_signatures, max_workers=8, chunksize=10)

# Filter 4.1: Presence of DE Genes
# Less than 50% of genes DE
# at least 50 DE genes
mask_de_genes = np.array(
    [(True if 0 <= len(d[0]) + len(d[1]) <= 1 * d[2] else False) for d in de_genes]
)

# get dsaids which pass filter
dsaids_f4 = np.array(dsaids_f3)[mask_de_genes]
print(f"Filtered by nº of DE genes: {len(dsaids_f4)}")

df_query = df_data_info_processed_filtered.query("dsaid in @dsaids_f4")
print(f"Filtering 4 - has 1 doid leaf")
print(f"Nº of signatures: {len(df_query)}")
print(f"Nº of datasets : {len(df_query['accession'].unique())}")
print(f"Nº of diseases : {len(df_query['disease'].unique())}")
print(
    f"Nº of disease ontology ids: {len(set(flatten([dsaids_2_doids[x] for x in dsaids_f4])))}"
)
print(
    f"Nº of disease ontology diseases: {len(set(flatten([list(set(dsaids_2_doids[x])&set(leaf_nodes)) for x in dsaids_f4])))}"
)
print(f"Nº of samples : {df_query['n_samples'].sum()}")

2025-08-02 16:45:41,396 - Nº of processed ids: 7191
2025-08-02 16:45:41,414 - Nº of processed ids in df_data_info: 7191
2025-08-02 16:45:41,416 - Nº of Filtered by library (filter out single cell): (7001, 12)


Nº of signatures: 6998
Nº of datasets : 3540
Nº of diseases : 1428
Nº of samples : 115102
Nº of DSAIDS with 1 leaf disease 2090
Filtering 3.1 - has doid
Nº of signatures: 5791
Nº of datasets : 3004
Nº of diseases : 912
Nº of disease ontology ids: 1286
Nº of disease ontology diseases: 838
Nº of samples : 94867
Filtering 3.2 - has doid leaf
Nº of signatures: 2893
Nº of datasets : 1606
Nº of diseases : 545
Nº of disease ontology ids: 993
Nº of disease ontology diseases: 838
Nº of samples : 41251
Filtering 3.3 - has 1 doid leaf
Nº of signatures: 2090
Nº of datasets : 1213
Nº of diseases : 460
Nº of disease ontology ids: 472
Nº of disease ontology diseases: 389
Nº of samples : 27689


100%|██████████| 2090/2090 [00:04<00:00, 501.25it/s]


Filtered by nº of DE genes: 2090
Filtering 4 - has 1 doid leaf
Nº of signatures: 2090
Nº of datasets : 1213
Nº of diseases : 460
Nº of disease ontology ids: 472
Nº of disease ontology diseases: 389
Nº of samples : 27689


In [6]:
# how good ise UMLS mappings to DOID?

# Build UMLS CUI → DOID mapping
umls_2_doid = {}

for node_id, data in do_G.nodes(data=True):
    xrefs = data.get('xref', [])
    for ref in xrefs:

        if ref.startswith('UMLS_CUI:'):
            umls_cui = ref.split(':', 1)[1]
            umls_2_doid[umls_cui] = node_id

# Example lookup
cui = 'C0002395'
doid = umls_2_doid.get(cui)

print(f'UMLS CUI {cui} maps to DOID: {doid}')



df_query_umls = df_data_info_processed_filtered.query("diseaseid in @umls_2_doid.keys()")
df_query_umls = df_query_umls.copy()
df_query_umls["doid"] = [
    umls_2_doid[x]
    for x in df_query_umls["diseaseid"]
]

UMLS CUI C0002395 maps to DOID: DOID:10652


In [55]:
# Compute Sanchez Information Content! 

import math
# Identify leaves (nodes with no children)
do_leaves = [n for n in do_G.nodes() if do_G.out_degree(n) == 0]
N_do_leaves = len(do_leaves)
print(f"Number of DO leaves: {N_do_leaves}")


# Compute IC for each node using Sánchez formula
do_sanchez_ic = {}

for node in do_G.nodes:
    # Descendant leaves
    descendants = nx.descendants(do_G, node)
    leaf_desc = [n for n in descendants if n in do_leaves]
    if node in do_leaves:
        leaf_desc.append(node)
    num_leaf_desc = len(leaf_desc)

    # Subsumers (ancestors + self)
    subsumers = nx.ancestors(do_G, node)
    subsumers.add(node)
    num_subsumers = len(subsumers)

    ic = -math.log((num_leaf_desc + 1) / ((num_subsumers + 1) * (N_do_leaves + 1)))
    do_sanchez_ic[node] = ic


# 1. Get the root node (most likely "DOID:4", but we’ll find it programmatically)
root_nodes = [n for n in do_G.nodes if do_G.in_degree(n) == 0]
assert len(root_nodes) == 1, "Multiple root nodes found!"
root = root_nodes[0]

# 2. Get level 1 and level 2 nodes
level1_nodes = list(do_G.successors(root))  # First level children
level2_nodes = [n2 for n1 in level1_nodes for n2 in do_G.successors(n1)]  # Second level

# 3. Combine and extract info
def get_node_info(node):
    return {
        "id": node,
        "name": do_G.nodes[node].get("name", "Unknown"),
        "sanchez_ic": do_sanchez_ic.get(node, None),
    }

# Get info for both levels
level1_info = [get_node_info(n) for n in level1_nodes]
level2_info = [get_node_info(n) for n in level2_nodes]

# Combine (optional)
all_info = {
    "level_1": level1_info,
    "level_2": level2_info,
}


pd.DataFrame(all_info["level_1"])

Number of DO leaves: 9511


,id,name,sanchez_ic
0,DOID:0014667,disease of metabolism,3.659051
1,DOID:0050117,disease by infectious agent,4.195137
2,DOID:0080015,physical disorder,3.999340
3,DOID:14566,disease of cellular proliferation,2.673133
4,DOID:150,disease of mental health,4.058413
5,DOID:225,syndrome,3.477864
6,DOID:630,genetic disease,1.828813
7,DOID:7,disease of anatomical entity,1.395447


In [56]:
pd.DataFrame(all_info["level_2"])

,id,name,sanchez_ic
0,DOID:0060158,acquired metabolic disease,6.739941
1,DOID:655,inherited metabolic disorder,4.228717
2,DOID:9120,amyloidosis,7.602165
3,DOID:104,bacterial infectious disease,5.477700
4,DOID:1398,parasitic infectious disease,6.003309
...,...,...,...
586,DOID:2914,immune system disease,3.944016
587,DOID:3463,breast disease,5.837074
588,DOID:74,hematopoietic system disease,4.525580
589,DOID:77,gastrointestinal system disease,4.131507


In [51]:
# Load DO data mappings
url = "http://purl.obolibrary.org/obo/doid.obo"
do_graph = obonet.read_obo(url)
do_G = nx.DiGraph(do_graph)

In [52]:
do_G = do_G.reverse()  # Reverse the graph to have root at the top

do_leaves = [n for n in do_G.nodes() if do_G.out_degree(n) == 0]
N_do_leaves = len(do_leaves)

print(f"Nº of DOID leaves: {N_do_leaves}")
print(f"DOID leaves: {do_leaves[:10]}")  # Print first 10 leaves for brevity

Nº of DOID leaves: 9511
DOID leaves: ['DOID:0040002', 'DOID:0040003', 'DOID:0040004', 'DOID:0040005', 'DOID:0040006', 'DOID:0040007', 'DOID:0040008', 'DOID:0040009', 'DOID:0040010', 'DOID:0040011']


In [54]:
node = "DOID:7"

# Descendant leaves
descendants = nx.descendants(do_G, node)
leaf_desc = [n for n in descendants if n in do_leaves]
if node in do_leaves:
    leaf_desc.append(node)
num_leaf_desc = len(leaf_desc)

# Subsumers (ancestors + self)
subsumers = nx.ancestors(do_G, node)
subsumers.add(node)
num_subsumers = len(subsumers)

ic = -math.log((num_leaf_desc + 1) / ((num_subsumers + 1) * (N_do_leaves + 1)))
print(num_leaf_desc, num_subsumers, N_do_leaves)
print(ic)

7068 2 9511
1.3954474208677734


In [41]:
descendants

set()

In [35]:
min(do_sanchez_ic.values())

0.6931471805599453

In [18]:
import scanpy as sc
import h5py

adata = sc.read_h5ad("/aloy/home/ddalton/projects/scGPT_playground/outputs/run-25-07-27-01/adata_train_1.h5ad")

celltypes_labels = adata.obs["celltype_id"].tolist()  # make sure count from 0
celltypes_labels = np.array(celltypes_labels)

In [ ]:
adata.obs["celltype_id"].nunique()

19

In [21]:
celltypes_labels

array([10,  6,  6, ..., 12,  6, 15])

In [20]:
celltypes_labels

array([10,  6,  6, ..., 12,  6, 15])

In [9]:
len(do_sanchez_ic)

11926

In [39]:
print(f"Nº of unique UMLS diseases {df_query_umls['diseaseid'].nunique()}",
      f"Nº of unique DOIDs {df_query_umls['doid'].nunique()}")

print(f"Nº of samples {df_query_umls['n_samples'].sum()}")

Nº of unique UMLS diseases 596 Nº of unique DOIDs 588
Nº of samples 77033


In [ ]:
# assess mappings
umls_uniq = list(df_query["diseaseid"].unique())
print(f"Nº of UMLS unique ids: {len(umls_uniq)}")
print(f"Nº of UMLS which map to DOID: {len(set(umls_uniq) & set(umls_2_doid.keys()))}")

# assess filtering by UMLS-> DOID mappings


Nº of UMLS unique ids: 415
Nº of UMLS which map to DOID: 295


In [13]:
df_data_info_processed_filtered.query("dsaid in @dsaids_w_diseases")["n_samples"].sum()

41251

In [10]:
df_query

,dsaid,accession,platform,deg_count,disease,diseaseid,tissue,data_source,library_strategy,organism,control_case_sample_count,definition,n_samples
3,DSA00004,GSE224022,GPL16791,1000,Retinoblastoma,C0035335,Retina,GEO,RNA-Seq,Homo sapiens,4|5,DO:A retinal cell cancer and malignant neoplas...,5
5,DSA00006,GSE126342,GPL11154,1000,Myotonic Dystrophy Type 1,C0027126,Skeletal muscle,GEO,RNA-Seq,Homo sapiens,6|16,DO:A myotonic disease that is characterized by...,16
49,DSA00050,GSE214376,GPL18573,1000,Sporadic Creutzfeldt-Jakob Disease,C1852467,Frontal cortex,GEO,RNA-Seq,Homo sapiens,9|5,MONDO:Inherited or familial Creutzfeldt-Jakob ...,5
70,DSA00071,GSE214987,GPL24676,1000,Osteoarthritis,C0157946,NaN,GEO,RNA-Seq,Homo sapiens,3|3,DO:An arthritis that has_material_basis_in wor...,3
71,DSA00072,GSE214987,GPL24676,142,Osteoarthritis,C0157946,NaN,GEO,RNA-Seq,Homo sapiens,3|3,DO:An arthritis that has_material_basis_in wor...,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10282,DSA10283,GSE27155,GPL96,1000,Papillary Thyroid Carcinoma,C0238463,Thyroid,GEO,Microarray,Homo sapiens,4|51,DO:An adenocarcinoma that derives_from epithel...,51
10283,DSA10284,GSE27155,GPL96,1000,Anaplastic Thyroid Carcinoma,C0238461,Thyroid,GEO,Microarray,Homo sapiens,4|2,DO:A thyroid gland carcinoma that is composed ...,2
10284,DSA10285,GSE27155,GPL96,1000,Anaplastic Thyroid Carcinoma,C0238461,Thyroid,GEO,Microarray,Homo sapiens,4|2,DO:A thyroid gland carcinoma that is composed ...,2
10286,DSA10287,GSE29272,GPL96,1000,Cancer of The Gastric Cardia,C1333763,Stomach,GEO,Microarray,Homo sapiens,134|72,MONDO:A carcinoma that arises from epithelial ...,72


In [ ]:
# Calculate the number of NaNs in each row
nan_counts = df.isna().sum(axis=1)

# # Filter the DataFrame to keep only rows with NaNs less than or equal to 18,000
# df = df[nan_counts <= 18000]
# logging.info(f"Filtered dataframe with shape: {df.shape}")


# Filter out Unknown samples
mask = [False if id.split(";")[2] == "Unknown" else True for id in df.iloc[:, 0].values]
df = df[mask]
logging.info(f"Filtered out Unkowns from dataframe with shape: {df.shape}")

2025-05-07 14:14:37,424 - Filtered out Unkowns from dataframe with shape: (50357, 19691)


In [ ]:
# endregion

# region 3. Convert to `adata` object
# Extract cell identifiers and gene expression data
ids = df.iloc[:, 0]
gene_expression_data = df.iloc[:, 1:].values
gene_names = df.columns[1:]

# Create an AnnData object
adata = ad.AnnData(X=gene_expression_data)

# Add cell and gene metadata
adata.obs["ids"] = ids.values

# gene symbols/name
adata.var["gene_symbols"] = gene_names
adata.var["gene_name"] = gene_names

# gene index - nomenclature scGPT
adata.var["index"] = gene_names

# get dataset
datasets = get_dataset(ids)
adata.obs["dataset"] = datasets

# get dataset
datasets = get_dataset(ids)
adata.obs["dataset_id"] = datasets

# get batch
dataset_accessions, batch_ids = get_dataset_to_batch(ids, df_info)
adata.obs["batch"] = batch_ids
adata.obs["batch_id"] = batch_ids

# get dsaid
dsaids = [x.split(";")[0] for x in ids]
adata.obs["dsaid"] = dsaids

# get tissues
tissues = get_tissue(ids)
adata.obs["tissue"] = tissues

# get nº genes
n_genes = (~np.isnan(adata.X)).sum(axis=1)
adata.obs["n_genes"] = n_genes

# get disease
diseases = get_disease(ids)
adata.obs["disease"] = diseases

# get celltype
diseases = get_disease(ids)
adata.obs["celltype"] = diseases

# get disease
diseases_study = get_disease_study(ids)
adata.obs["disease_study"] = diseases_study

# get library
library_stratergy = get_library(ids)
adata.obs["library"] = library_stratergy


if manual_parameters.get("ontology") == "mesh_id":
    # get mesh_ids
    mesh_id_study = [dsaid_2_mesh_id.get(id)[0] for id in dsaids]
    mesh_disease, mesh_id = get_mesh_disease(ids, mesh_id_2_term, dsaid_2_mesh_id)
    adata.obs["mesh_id_study"] = mesh_id_study
    adata.obs["mesh_id"] = mesh_id
    adata.obs["mesh_disease"] = mesh_disease

elif manual_parameters.get("ontology") == "doid":
    # get doid ids
    doid_study = [dsaids_2_doids_leaf.get(id)[0] for id in dsaids]
    doid_disease, doid_id = get_doid_disease(
        ids,
        doid_2_term,
        dsaids_2_doids_leaf,
    )
    adata.obs["do_id_study"] = doid_study

    adata.obs["do_id"] = doid_id
    adata.obs["do_term"] = doid_disease

    # add control with dataset label
    doid_id = [f"{x}-{y}" if x == "Control" else x for x, y in zip(doid_id, datasets)]
    adata.obs["do_id"] = doid_id    
    doid_term = [f"{x}-{y}" if x == "Control" else x for x, y in zip(doid_disease, datasets)]
    adata.obs["do_term"] = doid_term

In [ ]:
adata.obs

,ids,dataset,dataset_id,batch,batch_id,dsaid,tissue,n_genes,disease,celltype,disease_study,library,do_id_study,do_id,do_term
0,DSA00004;GSM7009973;Control,GSE224022,GSE224022,1027,1027,DSA00004,Retina,19399,Control,Control,Retinoblastoma,RNA-Seq,DOID:4648,Control-GSE224022,Control-GSE224022
1,DSA00004;GSM7009974;Control,GSE224022,GSE224022,1027,1027,DSA00004,Retina,19399,Control,Control,Retinoblastoma,RNA-Seq,DOID:4648,Control-GSE224022,Control-GSE224022
2,DSA00004;GSM7009976;Control,GSE224022,GSE224022,1027,1027,DSA00004,Retina,19399,Control,Control,Retinoblastoma,RNA-Seq,DOID:4648,Control-GSE224022,Control-GSE224022
3,DSA00004;GSM7009977;Control,GSE224022,GSE224022,1027,1027,DSA00004,Retina,19399,Control,Control,Retinoblastoma,RNA-Seq,DOID:4648,Control-GSE224022,Control-GSE224022
4,DSA00004;GSM7009978;Case,GSE224022,GSE224022,1027,1027,DSA00004,Retina,19399,Retinoblastoma,Retinoblastoma,Retinoblastoma,RNA-Seq,DOID:4648,DOID:4648,familial retinoblastoma
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50352,DSA10288;GSM723723;Case,GSE29272,GSE29272,1088,1088,DSA10288,Stomach,12002,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Microarray,DOID:6270,DOID:6270,gastric cardia carcinoma
50353,DSA10288;GSM723725;Case,GSE29272,GSE29272,1088,1088,DSA10288,Stomach,12002,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Microarray,DOID:6270,DOID:6270,gastric cardia carcinoma
50354,DSA10288;GSM723727;Case,GSE29272,GSE29272,1088,1088,DSA10288,Stomach,12002,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Microarray,DOID:6270,DOID:6270,gastric cardia carcinoma
50355,DSA10288;GSM723729;Case,GSE29272,GSE29272,1088,1088,DSA10288,Stomach,12002,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Microarray,DOID:6270,DOID:6270,gastric cardia carcinoma


In [ ]:
adata.obs["do_term"].to_list()[:100]

['familial retinoblastoma',
 'familial retinoblastoma',
 'familial retinoblastoma',
 'familial retinoblastoma',
 'familial retinoblastoma',
 'Control',
 'Control',
 'Control',
 'Control',
 'Control',
 'Control',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'myotonic dystrophy type 1',
 'Control',
 'Control',
 'Control',
 'Control',
 'Control',
 'Control',
 'Control',
 'Control',
 'Control',
 'Creutzfeldt-Jakob disease',
 'Creutzfeldt-Jakob disease',
 'Creutzfeldt-Jakob disease',
 'Creutzfeldt-Jakob disease',
 'Creutzfeldt-Jakob disease',
 'Control',
 'Control',
 'Control',
 'osteoarthrit

In [ ]:
if manual_parameters["dataset_exercise"] == "doid_dataset":
    # filter diseases with enough samples
    _filtered_doids = [
        k for k, v in adata.obs["do_id"].value_counts().to_dict().items() if v >= 5
    ]
    if "Control" in _filtered_doids:
        _filtered_doids.remove("Control")
    _filtered_doids = [d for d in _filtered_doids if not d.startswith("Control")]

    # filter samples
    _df_q = adata.obs.query("do_id_study in @_filtered_doids")
    logging.info(
        f"Filtered out doids with less than 5 samples from dataframe with shape: {_df_q.shape}"
    )

    # remove controls
    # _df_q = _df_q.query("do_id != 'Control'")
    # logging.info(f"Filtered out controls from dataframe with shape: {_df_q.shape}")

    # mask
    _mask = adata.obs["do_id"].isin(_filtered_doids)

    print(f"adata shape: {adata.shape}")

    print(f"Nº of DSAIDs: {adata.obs[_mask]['dsaid'].nunique()}")
    print(f"Nº Datasets: {adata.obs[_mask]['dataset'].nunique()}")
    print(f"Nº Diseases: {adata.obs[_mask]['disease'].nunique()}")
    print(f"Nº Samples: {(adata.obs[_mask]['disease'] != 'Control').sum()}")

    # filter adata
    adata = adata[_mask]
    logging.info(
        f"Filtered out doids with less than 5 samples  & Controls from dataframe with shape: {adata.shape}"
    )
    print(f"adata shape: {adata.shape}")

2025-05-07 15:15:07,679 - Filtered out doids with less than 5 samples from dataframe with shape: (49617, 15)
2025-05-07 15:15:07,721 - Filtered out doids with less than 5 samples  & Controls from dataframe with shape: (49108, 19690)


adata shape: (50357, 19690)
Nº of DSAIDs: 2017
Nº Datasets: 1144
Nº Diseases: 363
Nº Samples: 26868
adata shape: (49108, 19690)


In [24]:
# save to output file
output_folder = get_folder_name(base_output_dir)


# endregion

# region 4. Save to output file

# save adata
adata.write(os.path.join(output_folder, "data.h5ad"))

# save metadata
if diseases_of_interest_set is None:
    metadata_txt = "All Human Diseases - grouped disesases & no controls"
else:
    metadata_txt = ", ".join(diseases_of_interest_set)

# compute metadata values
n_genes = adata.X.shape[1]
n_gex = adata.X.shape[0]
n_non_nan_genes = np.sum(~np.isnan(adata.X), axis=0)
n_non_nan_gex = np.sum(~np.isnan(adata.X), axis=1)
genes_std = np.nanstd(adata.X, axis=0)
gex_std = np.nanstd(adata.X, axis=1)

# compute nº non-nan values per disease-dataset
all_dis_dt = [ds + ";" + dt for ds, dt in zip(diseases_study, datasets)]
unique_dis_dt = list(set(all_dis_dt))
gene_expression_data_bool = ~np.isnan(gene_expression_data)

n_non_nan_dis_dt_row = list()
n_non_nan_dis_dt_col = list()
for dis_dt in tqdm(unique_dis_dt):
    row_mask = np.isin(all_dis_dt, dis_dt)

    # get rows of interest
    rows_interest = gene_expression_data_bool[row_mask]

    # merge by columns
    merge_columns = rows_interest.sum(axis=0).astype(bool)

    # get nº non-nan values
    n_non_nan_values = merge_columns.sum()

    # append to list
    n_non_nan_dis_dt_row.append(n_non_nan_values)
    n_non_nan_dis_dt_col.append(merge_columns)

n_non_nan_dis_dt_genes = np.array(n_non_nan_dis_dt_col).sum(axis=0)

metadata = {
    "metadata": metadata_txt,
    "n_genes": n_genes,
    "n_gex": n_gex,
    "n_non_nan_genes": n_non_nan_genes,
    "n_non_nan_gex": n_non_nan_gex,
    "genes_std": genes_std,
    "gex_std": gex_std,
    "unique_dis_dt": unique_dis_dt,
    "n_non_nan_dis_dt_row": n_non_nan_dis_dt_row,
    "n_non_nan_dis_dt_genes": n_non_nan_dis_dt_genes,
}


metadata_path = os.path.join(output_folder, "metadata.pkl")
with open(metadata_path, "wb") as f:
    pickle.dump(metadata, f)

logging.info(f"Metadata saved to {metadata_path}")


# save manual parameters
# Write parameters to a JSON file
with open(os.path.join(output_folder, "parameters.json"), "w") as json_file:
    json.dump(manual_parameters, json_file, indent=4)


# endregion


# diseases_of_interest_set = {"Influenza", "Colorectal Carcinoma", "Asthma"}
# diseases_of_interest_set = None
# diseases_of_interest_set = {"Huntington's Disease", "Alzheimer's Disease", 'Asthma', 'COVID-19',
#        'Influenza', "Parkinson's Disease", 'Systemic Lupus Erythematosus',
#        'Obesity', 'Hepatocellular Carcinoma', "Crohn's Disease",
#        'Ulcerative Colitis', 'Sepsis', 'Breast Cancer', 'Psoriasis',
#        'Schizophrenia', 'Multiple Sclerosis', 'Amyotrophic Lateral Sclerosis',
#        'Tuberculosis', 'Chronic Obstructive Pulmonary Disease',
#        'Rheumatoid Arthritis', 'Idiopathic Pulmonary Fibrosis',
#        'Colorectal Carcinoma', 'Type 1 Diabetes',
#        'Non-Alcoholic Steatohepatitis', 'Melanoma', 'Diabetes',
#        'Myocardial Infarction', 'Acute Myeloid Leukemia (Aml-M2)', 'Colitis',
#        'Prostate Cancer'}

# diseases_of_interest_set = {'Acute-On-Chronic Liver Failure',
#  "Barrett's Esophagus",
#  "Behcet's Disease",
#  'Chronic Rhinosinusitis',
#  'Cornelia De Lange Syndrome',
#  'Coronary Artery Disease',
#  'Diabetes',
#  'Diabetic Kidney Disease',
#  'Follicular Lymphoma',
#  'Glioblastoma Multiforme',
#  'Hepatitis B',
#  'Hutchinson-Gilford Progeria Syndrome',
#  'Hypertension',
#  'Multiple System Atrophy',
#  'Pneumonia',
#  'Primary Myelofibrosis',
#  'Spinal Muscular Atrophy',
#  'Squamous Cell Carcinoma',
#  'Steatosis',
#  'Type 2 Diabetes Mellitus'}

# diseases_of_interest_set
# = {'Breast Cancer', 'Colorectal Carcinoma', 'Influenza'}
# diseases_of_interest_set = {'Control', 'Lung Adenocarcinoma', 'Breast Cancer', 'Psoriasis', 'Ulcerative Colitis', "Crohn's Disease", 'Lung Cancer'}

# diseases_of_interest_set = {
#     "Crohn's Disease",
#     "Ulcerative Colitis",
#     "Lung Cancer",
#     "Lung Adenocarcinoma",
#     "Breast Cancer",
#     "Psoriasis",
# }


#! LARGE DATASET - OLD
# QUERY = "library_strategy in @library_strategies_of_interest_set & organism == 'Homo sapiens'"
# dsaids_interest = np.array(df_info.query(QUERY)["dsaid"].to_list())
# size_df = len(pd.read_csv(large_df_path,usecols=["ID"]))

# logging.info(f"Reading merged dataframe {large_df_path}")

# list_filtered_df = list()

# for df_chunk in tqdm(pd.read_csv(large_df_path, chunksize=500), total=int(size_df/500)):
#     all_data_ids = df_chunk["ID"].to_list()
#     all_data_dsaids = np.array([id.split(";")[0] for id in all_data_ids])

#     logging.debug(f"all_data_ids: {all_data_ids}")
#     logging.debug(f"all_data_dsaids: {all_data_dsaids}")

#     mask = np.isin(all_data_dsaids,dsaids_interest)
#     logging.debug(f"mask {np.sum(mask)} : {mask}")
#     df_chunk_filtered = df_chunk[mask]

#     logging.debug(f"df_chunk_filtered {df_chunk_filtered}")

#     list_filtered_df.append(df_chunk_filtered)

# # merge filtered dataframes
# df = pd.concat(list_filtered_df)
# if "Unnamed: 0" in df.columns:
#     df.drop(columns=["Unnamed: 0"], inplace=True)


# variables
# manual_parameters = { "diseases_of_interest_set": list({
#     "Colorectal Carcinoma",
#     "Breast Cancer",
#     "Prostate Cancer",
#     "Hepatocellular Carcinoma",
#     "Crohn's Disease",
#     "Multiple Sclerosis"

# }),
#     "library_strategies_of_interest_set": list({
#         "Microarray"
#     }),
# }


# library_strategies_of_interest_set = {"RNA-Seq", "Microarray"}


# example_data_path = (
#     "/aloy/home/ddalton/projects/disease_signatures/data/DiSignAtlas/tmp/DSA00123.csv"
# )

... storing 'dataset' as categorical
... storing 'dataset_id' as categorical
... storing 'dsaid' as categorical
... storing 'tissue' as categorical
... storing 'disease' as categorical
... storing 'celltype' as categorical
... storing 'disease_study' as categorical
... storing 'library' as categorical
... storing 'do_id_study' as categorical
... storing 'do_id' as categorical
... storing 'do_term' as categorical


Output directory created: ../data/pp_data-25-05-07-01


/aloy/home/ddalton/miniconda3/envs/generic/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
100%|██████████| 1318/1318 [00:08<00:00, 147.86it/s]
2025-05-07 14:42:56,499 - Metadata saved to ../data/pp_data-25-05-07-01/metadata.pkl


2025-04-29 16:45:53,011 - Nº of DSAIDs of interest: 1222
2025-04-29 16:45:53,012 - Loaded dataframe with shape: (31342, 19691)
2025-04-29 16:45:54,023 - Filtered out Unkowns from dataframe with shape: (31342, 19691)
... storing 'dataset' as categorical
... storing 'dataset_id' as categorical
... storing 'dsaid' as categorical
... storing 'tissue' as categorical
... storing 'disease' as categorical
... storing 'celltype' as categorical
... storing 'disease_study' as categorical
... storing 'library' as categorical
... storing 'do_id_study' as categorical
... storing 'do_id' as categorical
... storing 'do_term' as categorical


Output directory created: ../data/pp_data-25-04-29-04


/aloy/home/ddalton/miniconda3/envs/generic/lib/python3.10/site-packages/numpy/lib/_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
100%|██████████| 859/859 [00:03<00:00, 237.73it/s]
2025-04-29 16:47:53,004 - Metadata saved to ../data/pp_data-25-04-29-04/metadata.pkl


In [ ]:
filtered_doids = [
    x for x, v in adata.obs["do_id"].value_counts().to_dict().items() if v > 50
]
adata.obs.query("do_id_study in @filtered_doids").shape

In [40]:
# Track all internal nodes and how many leaves they connect
from collections import defaultdict

# Load the ontology
url = "http://purl.obolibrary.org/obo/doid.obo"
do_graph = obonet.read_obo(url)
do_G = nx.DiGraph(do_graph)

# reverse the graph
reversed_graph = do_G.reverse()

leaf_nodes = [
    node
    for node in do_G.nodes()
    if do_G.in_degree(node) == 0 and do_G.out_degree(node) > 0
]

# map doid to term
doid_to_term = {
    node: data["name"] for node, data in do_G.nodes(data=True) if "name" in data
}


path_df_pairs = (
    "/aloy/home/ddalton/projects/BQ_diseases/outputs/do_pairs.universe.ic.csv"
)
# load all pairs of diseases from DO universe
df_pairs = pd.read_csv(path_df_pairs)

# filter only pairs in study
doids_study = adata.obs["do_id"].to_list()
df_query = df_pairs.query(f"node_1 in {doids_study} and node_2 in {doids_study}")
print(f"Pairs in Study: {len(df_query)}")

# filter most significant pairs based on IC Sanchez
pct = 1
doid_2_ic = dict(zip(df_pairs["node_lca"], df_pairs["ic_sanchez"]))
ic_values = [doid_2_ic[doid] for doid in df_pairs["node_lca"].unique()]

thr = np.percentile(ic_values, pct)

print(
    f"Non-Significant LCAs: {[doid_to_term[doid] for doid in df_query['node_lca'].unique() if doid_2_ic[doid] < thr]}"
)
non_significant_lcas = [
    doid for doid in df_query["node_lca"].unique() if doid_2_ic[doid] < thr
]


NameError: name 'adata' is not defined

In [48]:
thr = np.percentile(list(doid_2_ic.values()), 95)
print(thr)

doid_2_ic.get("DOID:65")

11.027475147535466


4.226971059300305

In [50]:
df_pairs

,node_1,node_2,shortest_path,pair_key,node_lca,term_lca,pair_key_doid,term_1,term_2,ic_seco,ic_sanchez,ic_zhou,ic_baseline
0,DOID:0040002,DOID:0040003,3,"('DOID:0060500', ('DOID:0040002', 'DOID:004000...",DOID:0060500,drug allergy,"('DOID:0040002', 'DOID:0040003')",aspirin allergy,benzylpenicillin allergy,0.578842,7.055612,0.559657,49
1,DOID:0040002,DOID:0040004,3,"('DOID:0060500', ('DOID:0040002', 'DOID:004000...",DOID:0060500,drug allergy,"('DOID:0040002', 'DOID:0040004')",aspirin allergy,amoxicillin allergy,0.578842,7.055612,0.559657,49
2,DOID:0040002,DOID:0040005,3,"('DOID:0060500', ('DOID:0040002', 'DOID:004000...",DOID:0060500,drug allergy,"('DOID:0040002', 'DOID:0040005')",aspirin allergy,ceftriaxone allergy,0.578842,7.055612,0.559657,49
3,DOID:0040002,DOID:0040006,2,"('DOID:0060500', ('DOID:0040002', 'DOID:004000...",DOID:0060500,drug allergy,"('DOID:0040002', 'DOID:0040006')",aspirin allergy,carbamazepine allergy,0.578842,7.055612,0.559657,49
4,DOID:0040002,DOID:0040007,2,"('DOID:0060500', ('DOID:0040002', 'DOID:004000...",DOID:0060500,drug allergy,"('DOID:0040002', 'DOID:0040007')",aspirin allergy,abacavir allergy,0.578842,7.055612,0.559657,49
...,...,...,...,...,...,...,...,...,...,...,...,...,...
48256865,DOID:9978,DOID:9986,7,"('DOID:7', ('DOID:9978', 'DOID:9986'))",DOID:7,disease of anatomical entity,"('DOID:9978', 'DOID:9986')",acute female pelvic peritonitis,orbit lymphoma,0.029927,1.395348,0.014959,7037
48256866,DOID:9978,DOID:9997,9,"('DOID:7', ('DOID:9978', 'DOID:9997'))",DOID:7,disease of anatomical entity,"('DOID:9978', 'DOID:9997')",acute female pelvic peritonitis,peripartum cardiomyopathy,0.029927,1.395348,0.014959,7037
48256867,DOID:998,DOID:9986,6,"('DOID:74', ('DOID:998', 'DOID:9986'))",DOID:74,hematopoietic system disease,"('DOID:998', 'DOID:9986')",eosinophilia-myalgia syndrome,orbit lymphoma,0.331583,4.523374,0.300908,411
48256868,DOID:998,DOID:9997,7,"('DOID:7', ('DOID:998', 'DOID:9997'))",DOID:7,disease of anatomical entity,"('DOID:998', 'DOID:9997')",eosinophilia-myalgia syndrome,peripartum cardiomyopathy,0.029927,1.395348,0.014959,7037


In [16]:

# get leaf nodes in dataset
ds_doids = list(adata.obs["do_id"].unique())

# leafs -> parents
ancestor_map = {leaf: nx.ancestors(reversed_graph, leaf) for leaf in leaf_nodes}

# parent -> leafs
parent_count = dict()
for leaf, ancestors in ancestor_map.items():
    for ancestor in ancestors:
        if ancestor not in parent_count:
            parent_count[ancestor] = set()
        parent_count[ancestor].add(leaf)


# Filter only those connecting to >1 leaf
connecting_nodes = {
    node: leaves for node, leaves in parent_count.items() if len(leaves) >= 5
}
print(f"Connecting nodes: {len(connecting_nodes)}")

Connecting nodes: 1045


In [17]:
# Compute IC for all internal nodes
internal_nodes = [n for n in do_G.nodes if n not in leaf_nodes]
ic_map = {n: doid_2_ic.get(n, 0) for n in internal_nodes}

# Map each leaf to its most informative ancestor
leaf_to_label = dict()
for leaf in ds_doids:
    if leaf not in ancestor_map:
        continue
    ancestors = ancestor_map[leaf]
    valid_ancestors = ancestors & set(ic_map.keys())
    valid_ancestors = ancestors & set(connecting_nodes.keys())
    valid_ancestors = ancestors - set(non_significant_lcas)
    if not valid_ancestors:
        continue
    # Pick the ancestor with the max IC value
    best_ancestor = min(valid_ancestors, key=lambda a: ic_map.get(a, 0))
    leaf_to_label[leaf] = best_ancestor

In [18]:
min_size = 5
from collections import Counter
valid_groups = [x for x, y in Counter(list(leaf_to_label.values())).items() if y >= min_size]

In [19]:
# list of diseases which are in valid groups
filtered_leafs = [x for x in leaf_to_label.keys() if leaf_to_label[x] in valid_groups]

print(f"Nº of leafs: {len(leaf_to_label)}")
print(f"Filtered leafs: {len(filtered_leafs)}")

Nº of leafs: 281
Filtered leafs: 211


In [20]:
# mask out leaf nodes which are not in filtered_groups
adata = adata[adata.obs["do_id"].isin(filtered_leafs)]

print(adata.shape)
# rename do_id to general node term
adata.obs["do_id"] = [
    leaf_to_label.get(x, x) for x in adata.obs["do_id"].to_list()
]
adata.obs["do_term"] = [
    doid_to_term.get(x, x) for x in adata.obs["do_id"].to_list()
]

adata.obs

(20932, 19690)


/tmp/ipykernel_8262/36819792.py:6: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["do_id"] = [


,ids,dataset,dataset_id,batch,batch_id,dsaid,tissue,n_genes,disease,celltype,disease_study,library,do_id_study,do_id,do_term
15,DSA00006;GSM3596890;Case,GSE126342,GSE126342,132,132,DSA00006,Skeletal muscle,19399,Myotonic Dystrophy Type 1,Myotonic Dystrophy Type 1,Myotonic Dystrophy Type 1,RNA-Seq,DOID:11722,DOID:0080000,muscular disease
16,DSA00006;GSM3596891;Case,GSE126342,GSE126342,132,132,DSA00006,Skeletal muscle,19399,Myotonic Dystrophy Type 1,Myotonic Dystrophy Type 1,Myotonic Dystrophy Type 1,RNA-Seq,DOID:11722,DOID:0080000,muscular disease
17,DSA00006;GSM3596892;Case,GSE126342,GSE126342,132,132,DSA00006,Skeletal muscle,19399,Myotonic Dystrophy Type 1,Myotonic Dystrophy Type 1,Myotonic Dystrophy Type 1,RNA-Seq,DOID:11722,DOID:0080000,muscular disease
18,DSA00006;GSM3596893;Case,GSE126342,GSE126342,132,132,DSA00006,Skeletal muscle,19399,Myotonic Dystrophy Type 1,Myotonic Dystrophy Type 1,Myotonic Dystrophy Type 1,RNA-Seq,DOID:11722,DOID:0080000,muscular disease
19,DSA00006;GSM3596894;Case,GSE126342,GSE126342,132,132,DSA00006,Skeletal muscle,19399,Myotonic Dystrophy Type 1,Myotonic Dystrophy Type 1,Myotonic Dystrophy Type 1,RNA-Seq,DOID:11722,DOID:0080000,muscular disease
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50352,DSA10288;GSM723723;Case,GSE29272,GSE29272,1088,1088,DSA10288,Stomach,12002,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Microarray,DOID:6270,DOID:77,gastrointestinal system disease
50353,DSA10288;GSM723725;Case,GSE29272,GSE29272,1088,1088,DSA10288,Stomach,12002,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Microarray,DOID:6270,DOID:77,gastrointestinal system disease
50354,DSA10288;GSM723727;Case,GSE29272,GSE29272,1088,1088,DSA10288,Stomach,12002,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Microarray,DOID:6270,DOID:77,gastrointestinal system disease
50355,DSA10288;GSM723729;Case,GSE29272,GSE29272,1088,1088,DSA10288,Stomach,12002,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Cancer of The Gastric Cardia,Microarray,DOID:6270,DOID:77,gastrointestinal system disease


In [23]:
adata.obs.groupby("do_term")["disease_study"].nunique()

do_term
X-linked monogenic disease           5
brain disease                       19
cardiovascular system disease       19
cell type cancer                     8
chromosomal disease                 10
cognitive disorder                   8
connective tissue disease           25
disease by infectious agent         26
disease of metabolism               13
gastrointestinal system disease     29
hematopoietic system disease         9
immune system cancer                11
integumentary system disease        16
muscular disease                    11
primary immunodeficiency disease    10
reproductive system disease         12
respiratory system disease          12
sensory system disease              10
urinary system disease               6
Name: disease_study, dtype: int64